# Pipeline Experimentation

Scratch work for the disease-module-discovery pipeline: expression-based node filtering trials, ID-space mapping trials (Ensembl/HGNC/UniProt/Entrez), external API exploration (g:Profiler, mygene.info, ProteomicsDB, UCSC Xena), and ad-hoc visualization tests.

This notebook is exploratory — code here informed the implementation in `diseasemodulediscovery/bin/tissue_specific_filtering.py`, but is not itself part of the pipeline or the thesis evaluation.

## 1 — Basic expression-based node filtering trials

In [ ]:
import pandas as pd 
import graph_tool.all as gt
import numpy as np 
gene_expression_by_tissue = pd.read_csv("./data/expression_by_tissue.gct", sep="\t", skiprows=2, header = 0)
expressed_genes = gene_expression_by_tissue[gene_expression_by_tissue["Bladder"] > 1]["Name"]
expressed_genes = list(gene.split(".")[0] for gene in expressed_genes)
print(expressed_genes[:10])


In [ ]:
network = gt.load_graph("./data/test_network.gt")
node_names = []
for v in network.vertices():
    node_names.append(network.vp.name[v])
gene_expression_by_tissue["Name"] = gene_expression_by_tissue["Name"].apply(lambda x: x.split(".")[0])
in_network_genes = gene_expression_by_tissue[gene_expression_by_tissue["Name"].isin(node_names)]
bladder_specific = in_network_genes[in_network_genes["Bladder"] > 1]["Name"]
bladder_specific = list(bladder_specific)





In [ ]:
import diseasemodulediscovery.bin.util as utils 
network = utils.load_graph('data/test_network.gt')
names = utils.name2index(network)
gene_expression_by_tissue = pd.read_csv("./data/expression_by_tissue.gct", sep="\t", skiprows=2, header = 0)
gene_expression_by_tissue["Name"] = gene_expression_by_tissue["Name"].apply(lambda x: x.split(".")[0])
gene_expression_by_tissue = gene_expression_by_tissue[gene_expression_by_tissue["Name"].isin(names.keys())]
gene_expression_by_tissue["node_id"] = gene_expression_by_tissue["Name"].map(names)
bladder_specific = gene_expression_by_tissue[gene_expression_by_tissue["Bladder"] > 1]
filter = network.new_vertex_property("bool")
filter.a[bladder_specific["node_id"]] = True
network.set_vertex_filter(filter)
network.purge_vertices()
network.clear_filters()
print(network.num_vertices())
gt.graph_draw(network, vertex_text=network.vp.name)


## 2 — ID space mapping trials

Ensembl ↔ HGNC ↔ UniProt ↔ Entrez, via pybiomart, g:Profiler, the UniProt idmapping file, and mygene.info.

In [ ]:
from pybiomart import Dataset

CHUNK_SIZE = 50

def chunks(values, chunk_size=CHUNK_SIZE):
    for start in range(0, len(values), chunk_size):
        yield values[start:start + chunk_size]

dataset_name = "hsapiens_gene_ensembl"
host = "www.ensembl.org"
dataset = Dataset(name=dataset_name, host=host)

results = []
for gene_chunk in chunks(expressed_genes):
    chunk_df = dataset.query(
        attributes=["ensembl_gene_id", "external_gene_name"],
        filters={"gene_id": gene_chunk},
    )
    if not chunk_df.empty:
        results.append(chunk_df)

if not results:
    raise ValueError("No genes found in the specified id space. Please check the id space and gene names.")

df = pd.concat(results, ignore_index=True).drop_duplicates()
print(df.head())


In [ ]:
from gprofiler import GProfiler

gp = GProfiler(return_dataframe=True)
gene_expression_by_tissue["Name"] = gene_expression_by_tissue["Name"].apply(lambda x: x.split(".")[0])
tissue_specific_expression = gene_expression_by_tissue[["Name", "Bladder"]]
print(tissue_specific_expression.shape)
df = gp.convert(organism="hsapiens", query=tissue_specific_expression["Name"].tolist(), target_namespace="HGNC")
print(df.shape)
df = df.merge(tissue_specific_expression, left_on="incoming", right_on="Name")
df = df[["Name", "converted", "Bladder"]]
not_found = df.loc[df["converted"].astype(str) == "None", "Name"].nunique()
print(not_found)
collapsed = (
    df[(df["converted"].astype(str) != "None")]
    .assign(name=lambda x : x["converted"].astype(str))
    .drop(columns=["Name"])
    .groupby("converted", as_index=False)
    .agg({
        "Bladder": "sum",
    })
)
print(collapsed.shape)
collapsed.head()



In [ ]:
df_uniprot = gp.convert(organism="hsapiens", query=tissue_specific_expression["Name"].tolist(), target_namespace="UNIPROT_GN_ACC")
print(df_uniprot.shape)
df_uniprot = df_uniprot.merge(tissue_specific_expression, left_on="incoming", right_on="Name")
df_uniprot = df_uniprot[["Name", "converted", "Bladder"]]
df_uniprot = df_uniprot.rename(columns={"converted":"UniProt"})
df_uniprot_valid = df_uniprot[df_uniprot["UniProt"].astype(str) != "None"].copy()
uniprot_mappings_per_gene = (
    df_uniprot_valid.assign(UniProt=lambda x: x["UniProt"].astype(str))
    .groupby("Name")["UniProt"]
    .nunique()
)
one_to_many_uniprot = uniprot_mappings_per_gene[uniprot_mappings_per_gene > 1]
print(f"Genes with one-to-many UniProt mappings: {len(one_to_many_uniprot)}")

In [ ]:
df_uniprot = (
    df_uniprot[(df_uniprot["UniProt"].astype(str) != "None")]
    .assign(UniProt=lambda x : x["UniProt"].astype(str))
    .groupby("UniProt", as_index=False)
    .agg({
        "Bladder": "sum",
        "Name" : lambda x : ";".join(x),
    })
)
print(df_uniprot.shape)
df_entrez = gp.convert(organism="hsapiens", query=df_uniprot["UniProt"].tolist(), target_namespace="ENTREZGENE_ACC")
df_entrez.head()
df_merged = df_entrez.merge(df_uniprot, left_on="incoming", right_on="UniProt")


In [ ]:
df_merged = df_merged.rename(columns={"converted":"Entrez", "Name":"source_ensembl_ids"})
df_merged = df_merged[["Entrez", "source_ensembl_ids", "Bladder"]]
mapping = (
    df_merged[df_merged["Entrez"].astype(str) != "None"]
    .assign(Entrez=lambda x: x["Entrez"].astype(str))
    .groupby("Entrez", as_index=False)
    .agg({
        "Bladder": "sum",
    })
)

print(mapping.shape)
mapping.head()

In [ ]:
uniprot_header = (
    "UniProtKB-AC",
    "UniProtKB-ID",
    "GeneID (EntrezGene)",
    "RefSeq",
    "GI",
    "PDB",
    "GO",
    "UniRef100",
    "UniRef90",
    "UniRef50",
    "UniParc",
    "PIR",
    "NCBI-taxon",
    "MIM",
    "UniGene",
    "PubMed",
    "EMBL",
    "EMBL-CDS",
    "Ensembl",
    "Ensembl_TRS",
    "Ensembl_PRO",
    "Additional PubMed",
)

uniprot_mapping = pd.read_csv(
    "./data/HUMAN_9606_idmapping_selected.tab",
    sep="\t",
    header=None,
    names=uniprot_header,
    dtype=str,
)

id_space_mapping = (
    uniprot_mapping[["UniProtKB-AC", "GeneID (EntrezGene)", "Ensembl"]]
    .rename(columns={"GeneID (EntrezGene)": "Entrez"})
    .assign(ensembl=lambda x: x["Ensembl"].str.split(";"))
    .explode("ensembl")
    .assign(ensembl=lambda x: x["ensembl"].str.split(".").str[0])
    .drop(columns="Ensembl")
    .dropna(subset=["ensembl"])
    .drop_duplicates()
)

gene_expression_id_mapping = (
    gene_expression_by_tissue.assign(Name=lambda x: x["Name"].str.split(".").str[0])
    .merge(id_space_mapping, left_on="Name", right_on="ensembl", how="inner")
    [["UniProtKB-AC", "Entrez", "ensembl"]]
    .drop_duplicates()
)

print(gene_expression_id_mapping.head())
print(gene_expression_id_mapping.shape)

nan_summary = gene_expression_id_mapping[["UniProtKB-AC", "Entrez"]].isna().sum()
nan_summary = nan_summary.to_frame(name="nan_count")
nan_summary["total_rows"] = len(gene_expression_id_mapping)
nan_summary["nan_fraction"] = nan_summary["nan_count"] / nan_summary["total_rows"]
print(nan_summary)


In [ ]:
import mygene 
mg = mygene.MyGeneInfo()
tissue_specific_expression = gene_expression_by_tissue[["Name", "Bladder"]]
tissue_specific_expression["Name"] = tissue_specific_expression["Name"].apply(lambda x: x.split(".")[0])
results = mg.querymany(
    tissue_specific_expression["Name"].tolist(),
    scopes="ensembl.gene",
    fields="symbol",
    species="human",
    as_dataframe=True,
    returnall=True,
    verbose=False,
)

In [ ]:
results["out"].head()

In [ ]:
res_df = results["out"].copy()

if "query" in res_df.columns:
    res_df = res_df.rename(columns={"query": "Name"})
else:
    res_df = res_df.reset_index().rename(columns={res_df.index.name or "index": "Name"})

res_df = res_df.rename(columns={"entrezgene": "Entrez"})

symbol_nan_count = res_df["symbol"].isna().sum()
res_df=res_df.dropna(subset=["symbol"])
res_df=res_df["symbol"].astype(str).unique()
print(f"NaN entries in Symbol column: {symbol_nan_count}")
print(len(res_df))


## 3 — Seed gene / first-neighbor conversion trials

In [ ]:
import pandas as pd 
alzheimer_genes = pd.read_csv("./data/alzheimer_seed_genes/genes.tsv", sep="\t", header=0)
alzheimer_genes.head()
alzheimer_genes["Gene"].to_csv("./data/alzheimer_seed_genes/symbol_ids.csv", index=False, header=False)

In [ ]:
import gprofiler
firstneighbor_test_nodes = pd.read_csv("data/firstneighbor.alzheimer_seed.nodes.tsv", header=0)
gp = gprofiler.GProfiler(return_dataframe=True)
gp_results = gp.convert(organism="hsapiens", query=firstneighbor_test_nodes["name"].tolist(), target_namespace="ENTREZGENE_ACC")
print(gp_results.loc[gp_results["converted"].astype(str) == "None", "incoming"].nunique())


## 4 — ProteomicsDB API exploration

Discovering the ProteomicsDB OData API surface (tissue list, entity schema) and pulling a small tissue-expression matrix with UniProt mapping.

In [ ]:
import requests
import pandas as pd

url = (
    "https://www.proteomicsdb.org/proteomicsdb/logic/api/"
    "tissuelist.xsodata/CA_AVAILABLEBIOLOGICALSOURCES_API"
)

params = {
    "select": ",".join([
        "TISSUE_ID",
        "TISSUE_NAME",
        "TISSUE_GROUP_NAME",
        "TISSUE_CATEGORY",
        "SCOPE_ID",
        "SCOPE_NAME",
        "QUANTIFICATION_METHOD_ID",
        "QUANTIFICATION_METHOD_NAME",
        "MS_LEVEL",
        "TAXCODE"
    ]),
    "$format": "json"
}

headers = {
    "Accept": "application/json"
}

r = requests.get(url, params=params, headers=headers)

print("Status code:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("First 500 characters:")
print(r.text[:500])

r.raise_for_status()

data = r.json()

rows = data.get("d", {}).get("results", data)

df = pd.DataFrame(rows)

df.head()

In [ ]:
human_tissues = df[
    (df["TAXCODE"] == 9606) &
    (df["TISSUE_CATEGORY"] == "tissue")
].copy()

tissue_list = (
    human_tissues[["TISSUE_ID", "TISSUE_NAME", "TISSUE_GROUP_NAME"]]
    .drop_duplicates()
    .sort_values("TISSUE_ID")
)

tissue_list.to_csv("proteomicsdb_tissue_list.tsv", sep="\t", index=False)

In [ ]:
BASE = "https://www.proteomicsdb.org/proteomicsdb/logic/api_v2/api.xsodata"

metadata_url = f"{BASE}/$metadata"

r = requests.get(metadata_url)

print("Status:", r.status_code)
print("Final URL:", r.url)
print("Content-Type:", r.headers.get("Content-Type"))
print(r.text[:500])

In [ ]:
import requests
import xml.etree.ElementTree as ET

root = ET.fromstring(r.text)

ns = {
    "edmx": "http://schemas.microsoft.com/ado/2007/06/edmx",
    "edm": "http://schemas.microsoft.com/ado/2008/09/edm",
}

# Map EntityType names to fields
entity_fields = {}

for entity in root.findall(".//edm:EntityType", ns):
    entity_name = entity.attrib["Name"]
    fields = [p.attrib["Name"] for p in entity.findall("edm:Property", ns)]
    entity_fields[entity_name] = fields

# Map EntitySet names to EntityTypes
for entity_set in root.findall(".//edm:EntitySet", ns):
    set_name = entity_set.attrib["Name"]
    entity_type = entity_set.attrib["EntityType"].split(".")[-1]

    fields = entity_fields.get(entity_type, [])

    if (
        "TISSUE_ID" in fields
        or "NORMALIZED_EXPRESSION" in fields
        or "UNIQUE_IDENTIFIER" in fields
    ):
        print("EntitySet name to use in URL:", set_name)
        print("EntityType:", entity_type)
        print("Fields:", fields)
        print()

In [ ]:
bto_ids = ["BTO:0000047", "BTO:0000233", "BTO:0000988"]

def get_tissue_expression_v2(bto_id):
    url = f"{BASE}/Tissue('{bto_id}')/ProteinExpression"

    r = requests.get(url, params={"$format": "json"})
    r.raise_for_status()

    data = r.json()
    rows = data.get("d", {}).get("results", data)

    df = pd.DataFrame(rows)
    df = df.drop(columns=["__metadata"], errors="ignore")

    df = df[["PROTEIN_ID", "EXPRESSION"]].copy()
    df["EXPRESSION"] = pd.to_numeric(
        df["EXPRESSION"],
        errors="coerce"
    )
    # make sure one protein only has one value per tissue
    df = (
        df.groupby("PROTEIN_ID", as_index=False)["EXPRESSION"]
        .mean()
    )

    df = df.rename(columns={"EXPRESSION": bto_id})

    return df

all_tissue_dfs = []

for bto_id in bto_ids: 
    print(f"Querying {bto_id}")
    try:
        df_tissue = get_tissue_expression_v2(bto_id=bto_id)
        all_tissue_dfs.append(df_tissue)
    except Exception as e: 
        print(f"skipping {bto_id}: {e}")

In [ ]:
from functools import reduce 
expression_matrix = reduce(
    lambda left, right : pd.merge(left, right, on="PROTEIN_ID", how="outer"),
    all_tissue_dfs
)
expression_matrix.fillna(value=0,inplace=True)
expression_matrix.head()


In [ ]:
example_id = int(expression_matrix["PROTEIN_ID"].dropna().iloc[0])

url = f"{BASE}/Protein({example_id})"

r = requests.get(url, params={"$format": "json"})
r.raise_for_status()

example_protein = r.json().get("d", r.json())

example_protein

In [ ]:
def get_protein_mapping(protein_id):
    url = f"{BASE}/Protein({int(protein_id)})"

    r = requests.get(url, params={"$format": "json"})
    r.raise_for_status()

    data = r.json().get("d", r.json())

    return {
        "PROTEIN_ID": int(protein_id),

        # Try common possible UniProt fields
        "UNIPROT_ID": (
            data.get("UNIQUE_IDENTIFIER")
            or data.get("UNIPROT_ID")
            or data.get("UNIPROT_ACCESSION")
            or data.get("ACCESSION")
        ),

        "DATABASE": data.get("DATABASE"),
        "ENTRY_NAME": data.get("ENTRY_NAME"),
        "GENE_NAME": data.get("GENE_NAME"),
        "PROTEIN_DESCRIPTION": data.get("PROTEIN_DESCRIPTION"),
    }

In [ ]:

protein_ids = (
    expression_matrix["PROTEIN_ID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

mapping_rows = []

for protein_id in protein_ids:
    try:
        mapping_rows.append(get_protein_mapping(protein_id))
    except Exception as e:
        print(f"Could not map PROTEIN_ID {protein_id}: {e}")

protein_mapping = pd.DataFrame(mapping_rows)

protein_mapping.head()

In [ ]:
expression_matrix_uniprot = protein_mapping.merge(
    expression_matrix,
    on="PROTEIN_ID",
    how="right"
)

expression_matrix_uniprot.head()
print(expression_matrix_uniprot.shape)

In [ ]:
# Keep only rows where a UniProt ID was found
expression_matrix_uniprot = expression_matrix_uniprot.dropna(
    subset=["UNIPROT_ID"]
).copy()
print(expression_matrix_uniprot.shape)

# Optional: if multiple ProteomicsDB IDs map to the same UniProt ID,
# average their expression values
metadata_cols = [
    "UNIPROT_ID",
    "DATABASE",
    "ENTRY_NAME",
    "GENE_NAME",
    "PROTEIN_DESCRIPTION",
    "PROTEIN_ID"
]

expression_cols = [
    col for col in expression_matrix_uniprot.columns
    if col not in metadata_cols
]

expression_matrix_uniprot_final = (
    expression_matrix_uniprot
    .groupby("UNIPROT_ID", as_index=False)[expression_cols]
    .mean()
)

expression_matrix_uniprot_final.head()

## 5 — Expression-colored graph visualization trial

In [ ]:
import graph_tool.all as gt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np

# -----------------------------
# 1. Generate toy graph
# -----------------------------
g = gt.collection.data["karate"].copy()

# -----------------------------
# 2. Create random expression values
# -----------------------------
expr = g.new_vertex_property("double")

rng = np.random.default_rng(seed=42)

for v in g.vertices():
    expr[v] = rng.uniform(0, 10)  # random expression between 0 and 10

g.vp.expression = expr

# -----------------------------
# 3. Map expression values to colors
# -----------------------------
values = expr.a

norm = mcolors.Normalize(
    vmin=values.min(),
    vmax=values.max()
)

cmap = cm.get_cmap("Reds")  # low = light red, high = dark red

vertex_fill_color = g.new_vertex_property("vector<double>")

for v in g.vertices():
    rgba = cmap(norm(expr[v]))
    vertex_fill_color[v] = rgba

# -----------------------------
# 4. Draw graph
# -----------------------------
pos = gt.sfdp_layout(g)

gt.graph_draw(
    g,
    pos=pos,
    vertex_fill_color=vertex_fill_color,
    vertex_color="black",
    vertex_size=18,
    edge_color=[0.7, 0.7, 0.7, 0.5],
    output_size=(700, 700),
    output="toy_expression_graph.pdf"
)

## 6 — UCSC Xena / TCGA API trial

In [ ]:
import xenaPython as xena

hub = xena.PUBLIC_HUBS["gdcHub"]
cohort = "GDC TCGA Lung Adenocarcinoma (LUAD)"   # the actual Xena cohort label
cohorts = xena.all_cohorts(hub, [])
print(cohorts)
cohort = [cohort for cohort in cohorts if "TCGA" in cohort and "LUAD" in cohort][0]
all_datasets = xena.dataset_list(hub, [cohort])
for d in all_datasets:
    print(d["name"])
tpm_dataset = [d["name"] for d in all_datasets if "star_tpm" in d["name"]]

print(tpm_dataset)

In [ ]:
import requests, gzip, io
import pandas as pd
dataset_id = tpm_dataset[0]
url = f"{hub}/download/{dataset_id}.gz"

resp = requests.get(url, stream=True)
resp.raise_for_status()

with gzip.GzipFile(fileobj=io.BytesIO(resp.content)) as gz:
    expr = pd.read_csv(gz, sep="\t", index_col=0)

print(expr.shape)      # genes x samples
expr.head()